In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import os
import plotly.express as px

In [44]:
pl_24_25 = pd.read_csv("../data/processed/understat/ENG-Premier League/2024-2025/team_season.csv")
pl_25_26 = pd.read_csv("../data/processed/understat/ENG-Premier League/2025-2026/team_season.csv")
team_match = pd.read_csv("../data/processed/understat/ENG-Premier League/2025-2026/team_match.csv")

combined = pl_24_25.merge(
    pl_25_26,
    on="team_id",
    how="inner",
    suffixes=("_24_25", "_25_26"),
    validate="one_to_one"
)

combined = combined.rename(columns = {"league_24_25": 'league',"league_id_24_25": 'league',"team_24_25" :"team"})
display(combined)


,league,season_24_25,league,season_id_24_25,team,team_id,understat_team_id_24_25,matches_24_25,home_matches_24_25,away_matches_24_25,...,xga_25_26,xg_difference_25_26,np_xg_25_26,np_xga_25_26,np_xg_against_25_26,np_xg_difference_25_26,deep_completions_25_26,deep_completions_allowed_25_26,ppda_25_26,ppda_allowed_25_26
0,ENG-Premier League,2024-2025,1,2024,LIV,259f237e,87,38,19,19,...,53.869802,13.388930,65.736392,50.825129,50.825129,14.911263,373,214,10.539527,14.744389
1,ENG-Premier League,2024-2025,1,2024,ARS,1dd1f33c,83,38,19,19,...,33.133805,44.352808,74.441933,33.133805,33.133805,41.308128,349,142,10.325832,15.205794
2,ENG-Premier League,2024-2025,1,2024,MCI,b11c2f05,88,38,19,19,...,46.492602,32.866953,76.314868,44.820279,44.820279,31.494589,409,162,11.547097,14.782724
3,ENG-Premier League,2024-2025,1,2024,CHE,1464d629,80,38,19,19,...,58.117463,14.094997,66.123101,56.595126,56.595126,9.527975,273,252,10.618019,17.233087
4,ENG-Premier League,2024-2025,1,2024,NEW,ddbd77d6,86,38,19,19,...,57.312037,3.301212,56.046229,53.506191,53.506191,2.540038,274,280,10.699201,12.218486
5,ENG-Premier League,2024-2025,1,2024,AVL,bcc4ae28,71,38,19,19,...,56.682839,-0.479528,56.203311,55.160499,55.160499,1.042812,293,217,13.049547,10.937199
6,ENG-Premier League,2024-2025,1,2024,NFO,03a0d347,249,38,19,19,...,64.739616,-15.803370,46.652752,60.172594,60.172594,-13.519842,210,294,14.889914,10.191859
7,ENG-Premier League,2024-2025,1,2024,BHA,5dbeea62,220,38,19,19,...,53.751226,4.668453,54.613838,49.184214,49.184214,5.429624,270,251,9.074002,15.436909
8,ENG-Premier League,2024-2025,1,2024,BOU,56f0abca,73,38,19,19,...,56.769789,10.064763,62.267542,52.202807,52.202807,10.064735,275,274,10.676201,10.939113
9,ENG-Premier League,2024-2025,1,2024,BRE,8331d109,244,38,19,19,...,59.110111,7.146563,58.644998,53.020759,53.020759,5.624239,240,278,11.640723,12.110689


In [45]:
print(combined.columns)

Index(['league', 'season_24_25', 'league', 'season_id_24_25', 'team',
       'team_id', 'understat_team_id_24_25', 'matches_24_25',
       'home_matches_24_25', 'away_matches_24_25', 'wins_24_25', 'draws_24_25',
       'losses_24_25', 'points_24_25', 'expected_points_24_25',
       'goals_for_24_25', 'goals_against_24_25', 'goal_difference_24_25',
       'xg_24_25', 'xga_24_25', 'xg_difference_24_25', 'np_xg_24_25',
       'np_xga_24_25', 'np_xg_against_24_25', 'np_xg_difference_24_25',
       'deep_completions_24_25', 'deep_completions_allowed_24_25',
       'ppda_24_25', 'ppda_allowed_24_25', 'league_25_26', 'season_25_26',
       'league_id_25_26', 'season_id_25_26', 'team_25_26',
       'understat_team_id_25_26', 'matches_25_26', 'home_matches_25_26',
       'away_matches_25_26', 'wins_25_26', 'draws_25_26', 'losses_25_26',
       'points_25_26', 'expected_points_25_26', 'goals_for_25_26',
       'goals_against_25_26', 'goal_difference_25_26', 'xg_25_26', 'xga_25_26',
       'xg_di

In [46]:
# Points Swing Calculation
pts_swing = combined[['team', 'team_id', 'points_24_25', 'points_25_26']].copy()

pts_swing['pts_swing'] = pts_swing['points_25_26'] - pts_swing['points_24_25']

pts_swing = pts_swing.sort_values("pts_swing", ascending=False)

pts_swing["swing_direction"] = np.where(
    pts_swing["pts_swing"] >= 0,
    "Positive",
    "Negative"
)

fig = px.bar(
    pts_swing,
    x="pts_swing",
    y="team",
    text="pts_swing",
    color="swing_direction",
    color_discrete_map={
        "Positive": "#2E7D32",
        "Negative": "#C62828",
    },
    category_orders={
        "team": pts_swing["team"].tolist()
    }
)

fig.update_layout(
    height=600,
    width=1000,
    title={
        "text": "Premier League Points Swing 24/25 to 25/26",
        "x": 0.5,
        "xanchor": "center",
        "yanchor": "top",
        "font": {"size": 20, "color": "#3A2A1A"},
    },
    font={
        "family": "Gill Sans, sans-serif",
        "size": 11,
        "color": "#3A2A1A",
    },
    margin={"l": 40, "r": 40, "t": 70, "b": 60},
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    legend={
        "font": {"color": "#3A2A1A"},
        "title": {"font": {"color": "#3A2A1A"}},
    },
    xaxis={
        "showgrid": False,
        "zeroline": False,
        "linecolor": "#5A4632",
        "tickcolor": "#5A4632",
        "tickfont": {"size": 13, "color": "#5A4632"},
        "title": {"font": {"color": "#3A2A1A"}, "text": "Points Swing"},
    },
    yaxis={
        "showgrid": False,
        "gridcolor": "#D8C8AF",
        "zeroline": False,
        "linecolor": "#5A4632",
        "tickcolor": "#5A4632",
        "tickfont": {"size": 13, "color": "#5A4632"},
        "title": {"font": {"color": "#3A2A1A"}, "text": "Team"},
        "categoryorder": "array",
        "categoryarray": pts_swing["team"].tolist(),
    },
)
fig.add_traces

fig.show()

In [47]:
print(team_match.columns)
print(team_match.dtypes)


Index(['league', 'season', 'league_id', 'season_id', 'game_id', 'game_date',
       'game_time', 'game', 'team', 'result', 'round', 'team_id',
       'understat_team_id', 'opp', 'opp_id', 'understat_opp_id', 'venue',
       'points', 'expected_points', 'goals', 'goals_against', 'xg', 'xga',
       'np_xg', 'np_xga', 'np_xg_difference', 'wins', 'draws', 'losses',
       'ppda', 'ppda_allowed', 'deep_completions', 'deep_completions_allowed',
       'team_start_elo', 'team_end_elo', 'opp_start_elo', 'opp_end_elo',
       'elo_diff_start', 'elo_diff_end'],
      dtype='str')
league                          str
season                          str
league_id                     int64
season_id                     int64
game_id                       int64
game_date                       str
game_time                       str
game                            str
team                            str
result                          str
round                         int64
team_id                   

In [48]:
team_dr = team_match['team_start_elo'] - team_match['opp_start_elo']
opp_dr = team_match['opp_start_elo'] - team_match['team_start_elo']
team_match['exp_result'] = 1 / (10 ** (- team_dr / 400) + 1)
team_match['opp_exp_result'] = 1 / (10 ** (- opp_dr / 400) + 1)

In [58]:
def add_tilt(team_match, initial_tilt=1.0, weight=0.02):
    df = team_match.copy()
    df['game_date'] = pd.to_datetime(df['game_date'])
    df = df.sort_values(['game_date', 'game_id', 'team']).copy()

    current_tilts = {}
    start_tilts = {}
    opp_start_tilts = {}
    end_tilts = {}

    for game_id, game in df.groupby('game_id', sort=False):
        # Snapshot before updating either team in this fixture
        pre_match_tilts = {
            row['team']: current_tilts.get(row['team'], initial_tilt)
            for _, row in game.iterrows()
        }

        updates = {}

        for idx, row in game.iterrows():
            team = row['team']
            opponent = row['opp']

            old_tilt = current_tilts.get(team, initial_tilt)
            opposition_tilt = current_tilts.get(opponent, initial_tilt)

            game_total_goals = row['goals'] + row['goals_against']

            # Placeholder expected total goals.
            # Replace this with your Elo-difference expected-goals model later.
            exp_game_total_goals = row['xg'] + row['xga']

            if exp_game_total_goals > 0:
                new_tilt = (
                    (1 - weight) * old_tilt
                    + weight * (game_total_goals / opposition_tilt / exp_game_total_goals)
                )
            else:
                new_tilt = old_tilt

            start_tilts[idx] = old_tilt
            opp_start_tilts[idx] = opposition_tilt
            end_tilts[idx] = new_tilt
            updates[team] = new_tilt

        # Apply fixture updates only after all rows used pre-match tilt values
        current_tilts.update(updates)

    df['team_start_tilt'] = df.index.map(start_tilts)
    df['opp_start_tilt'] = df.index.map(opp_start_tilts)
    df['team_end_tilt'] = df.index.map(end_tilts)

    return df

team_match = add_tilt(team_match)
team_match = team_match.sort_values('team').copy()

In [59]:
display(team_match)

,league,season,league_id,season_id,game_id,game_date,game_time,game,team,result,...,xg_90,rolling_xg_rank,avg_rolling_xg_rank,avg_xg_rank,rolling_xga,tot_xga,avg_xga,xga_90,rolling_xga_rank,avg_xga_rank
21,ENG-Premier League,2025-2026,1,2025,28994,2026-01-17,17:30:00,NFO - ARS,ARS,D,...,0.860962,1.0,2.0,2.0,1.047397,33.133805,0.871942,0.368153,3.0,1.0
9,ENG-Premier League,2025-2026,1,2025,28869,2025-11-01,15:00:00,BUR - ARS,ARS,W,...,0.860962,5.0,2.0,2.0,0.627289,33.133805,0.871942,0.368153,1.0,1.0
17,ENG-Premier League,2025-2026,1,2025,28948,2025-12-27,15:00:00,ARS - BHA,ARS,W,...,0.860962,4.0,2.0,2.0,0.970377,33.133805,0.871942,0.368153,2.0,1.0
2,ENG-Premier League,2025-2026,1,2025,28806,2025-08-31,15:30:00,LIV - ARS,ARS,L,...,0.860962,7.0,2.0,2.0,0.671565,33.133805,0.871942,0.368153,2.0,1.0
23,ENG-Premier League,2025-2026,1,2025,29015,2026-01-31,15:00:00,LEE - ARS,ARS,W,...,0.860962,4.0,2.0,2.0,0.480916,33.133805,0.871942,0.368153,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
727,ENG-Premier League,2025-2026,1,2025,28835,2025-09-27,19:00:00,TOT - WOL,WOL,D,...,0.427127,17.0,20.0,19.0,1.258018,68.303994,1.797474,0.758933,6.0,19.0
744,ENG-Premier League,2025-2026,1,2025,29005,2026-01-24,15:00:00,MCI - WOL,WOL,L,...,0.427127,18.0,20.0,19.0,0.976132,68.303994,1.797474,0.758933,2.0,19.0
758,ENG-Premier League,2025-2026,1,2025,29147,2026-05-17,15:00:00,WOL - FUL,WOL,D,...,0.427127,19.0,20.0,19.0,1.850883,68.303994,1.797474,0.758933,13.0,19.0
726,ENG-Premier League,2025-2026,1,2025,28820,2025-09-20,14:00:00,WOL - LEE,WOL,L,...,0.427127,20.0,20.0,19.0,1.554082,68.303994,1.797474,0.758933,16.0,19.0


In [60]:
amorim_era = team_match[team_match['game_date'] < '2026-01-05']
carrick_era = team_match[team_match['game_date'] > '2026-01-05']

In [61]:
print(amorim_era['game_id'].nunique())

200


In [62]:
amorim_era['rolling_xg'] = amorim_era.groupby('team')['xg'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True).round(2)
carrick_era['rolling_xg'] = carrick_era.groupby('team')['xg'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True).round(2)

amorim_era['tot_xg'] = amorim_era.groupby('team')['xg'].transform(lambda x: x.sum()).round(2)
carrick_era['tot_xg'] = carrick_era.groupby('team')['xg'].transform(lambda x: x.sum()).round(2)

amorim_era['avg_xg'] = amorim_era.groupby('team')['xg'].transform(lambda x: x.mean()).round(2)
carrick_era['avg_xg'] = carrick_era.groupby('team')['xg'].transform(lambda x: x.mean()).round(2)

amorim_era['avg_rolling_xg'] = amorim_era.groupby('team')['rolling_xg'].transform(lambda x: x.mean()).round(2)
carrick_era['avg_rolling_xg'] = carrick_era.groupby('team')['rolling_xg'].transform(lambda x: x.mean()).round(2)

amorim_era['xg_90'] = amorim_era.groupby('round')['tot_xg'].transform(lambda x: x / 90).round(2)
carrick_era['xg_90'] = carrick_era.groupby('team')['tot_xg'].transform(lambda x: x / 90).round(2)

amorim_era['rolling_xg_rank'] = amorim_era.groupby('round')['rolling_xg'].rank(ascending=False).astype(int)
carrick_era['rolling_xg_rank'] = carrick_era.groupby('round')['rolling_xg'].rank(ascending=False).astype(int)

amorim_era['avg_rolling_xg_rank'] = amorim_era.groupby('round')['avg_rolling_xg'].rank(ascending=False).astype(int)
carrick_era['avg_rolling_xg_rank'] = carrick_era.groupby('round')['avg_rolling_xg'].rank(ascending=False).astype(int)

amorim_era['avg_xg_rank'] = amorim_era.groupby('round')['avg_xg'].rank(ascending=False).astype(int)
carrick_era['avg_xg_rank'] = carrick_era.groupby('round')['avg_xg'].rank(ascending=False).astype(int)

#Goals Against
amorim_era['rolling_xga'] = amorim_era.groupby('team')['xga'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True).round(2)
carrick_era['rolling_xga'] = carrick_era.groupby('team')['xga'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True).round(2)

amorim_era['tot_xga'] = amorim_era.groupby('team')['xga'].transform(lambda x: x.sum()).round(2)
carrick_era['tot_xga'] = carrick_era.groupby('team')['xga'].transform(lambda x: x.sum()).round(2)

amorim_era['avg_xga'] = amorim_era.groupby('team')['xga'].transform(lambda x: x.mean()).round(2)
carrick_era['avg_xga'] = carrick_era.groupby('team')['xga'].transform(lambda x: x.mean()).round(2)

amorim_era['tilt'] = amorim_era.groupby('team')['team_end_tilt'].transform(lambda x: x.mean()).round(2)
carrick_era['tilt'] = carrick_era.groupby('team')['team_end_tilt'].transform(lambda x: x.mean()).round(2)

amorim_era['xga_90'] = amorim_era.groupby('round')['tot_xga'].transform(lambda x: x / 90).round(2)
carrick_era['xga_90'] = carrick_era.groupby('team')['tot_xga'].transform(lambda x: x / 90).round(2)

amorim_era['rolling_xga_rank'] = amorim_era.groupby('round')['rolling_xga'].rank(ascending=True).astype(int)
carrick_era['rolling_xga_rank'] = carrick_era.groupby('round')['rolling_xga'].rank(ascending=True).astype(int)

amorim_era['avg_xga_rank'] = amorim_era.groupby('round')['avg_xga'].rank(ascending=True).astype(int)
carrick_era['avg_xga_rank'] = carrick_era.groupby('round')['avg_xga'].rank(ascending=True).astype(int)

display(amorim_era[['team','opp', 'game', 'round', 'game_date','rolling_xg', 'rolling_xg_rank', 'avg_rolling_xg', 'avg_rolling_xg_rank','tot_xg', 'avg_xg', 'xg_90', 'avg_xg_rank', 'rolling_xga', 'rolling_xga_rank', 'tot_xga', 'avg_xga', 'xga_90', 'avg_xga_rank','team_start_elo', 'team_end_elo', 'opp_start_elo', 'opp_end_elo', 'exp_result', 'opp_exp_result', 'tilt']])
display(carrick_era[['team','opp', 'game', 'round', 'game_date','rolling_xg', 'rolling_xg_rank', 'avg_rolling_xg', 'avg_rolling_xg_rank', 'tot_xg', 'avg_xg','xg_90', 'avg_xg_rank', 'rolling_xga', 'rolling_xga_rank', 'tot_xga', 'avg_xga', 'xga_90', 'avg_xga_rank', 'team_start_elo', 'team_end_elo', 'opp_start_elo', 'opp_end_elo', 'exp_result', 'opp_exp_result', 'tilt']])


,team,opp,game,round,game_date,rolling_xg,rolling_xg_rank,avg_rolling_xg,avg_rolling_xg_rank,tot_xg,...,avg_xga,xga_90,avg_xga_rank,team_start_elo,team_end_elo,opp_start_elo,opp_end_elo,exp_result,opp_exp_result,tilt
9,ARS,BUR,BUR - ARS,10,2025-11-01,2.52,1,2.16,1,40.76,...,0.87,0.19,1,2032.833740,2036.797852,1734.798584,1730.834473,0.847565,0.152435,0.99
17,ARS,BHA,ARS - BHA,18,2025-12-27,2.53,1,2.16,1,40.76,...,0.87,0.19,1,2044.442139,2046.828491,1832.210449,1832.210449,0.772363,0.227637,0.99
2,ARS,LIV,LIV - ARS,3,2025-08-31,1.85,7,2.16,1,40.76,...,0.87,0.19,1,2002.376343,1995.595581,2003.530518,2010.311279,0.498339,0.501661,0.99
1,ARS,LEE,ARS - LEE,2,2025-08-23,2.07,3,2.16,1,40.76,...,0.87,0.19,1,1997.870850,2002.189209,1731.203125,1726.884766,0.822746,0.177254,0.99
10,ARS,SUN,SUN - ARS,11,2025-11-08,2.12,3,2.16,1,40.76,...,0.87,0.19,1,2046.106567,2038.407349,1642.506714,1650.205811,0.910789,0.089211,0.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
741,WOL,WHU,WOL - WHU,20,2026-01-03,1.41,14,0.87,20,18.66,...,1.62,0.36,18,1647.172607,1661.810425,1722.781860,1708.144165,0.392875,0.607125,1.02
729,WOL,SUN,SUN - WOL,8,2025-10-18,1.38,12,0.87,20,18.66,...,1.62,0.36,18,1709.857422,1696.114746,1602.296509,1616.039185,0.650030,0.349970,1.02
728,WOL,BHA,WOL - BHA,7,2025-10-05,1.42,10,0.87,20,18.66,...,1.62,0.36,18,1707.453857,1709.857422,1834.567017,1834.567017,0.324817,0.675183,1.02
727,WOL,TOT,TOT - WOL,6,2025-09-27,1.13,14,0.87,20,18.66,...,1.62,0.36,18,1702.591919,1706.817505,1816.759766,1812.534058,0.341368,0.658632,1.02


,team,opp,game,round,game_date,rolling_xg,rolling_xg_rank,avg_rolling_xg,avg_rolling_xg_rank,tot_xg,...,avg_xga,xga_90,avg_xga_rank,team_start_elo,team_end_elo,opp_start_elo,opp_end_elo,exp_result,opp_exp_result,tilt
21,ARS,NFO,NFO - ARS,22,2026-01-17,2.86,1,2.21,1,36.72,...,0.88,0.18,1,2052.350586,2046.634888,1767.032715,1772.748291,0.837864,0.162136,1.00
23,ARS,LEE,LEE - ARS,24,2026-01-31,3.20,1,2.21,1,36.72,...,0.88,0.18,1,2056.450195,2062.516602,1775.709229,1769.642578,0.834253,0.165747,1.00
34,ARS,FUL,ARS - FUL,35,2026-05-02,3.50,1,2.21,1,36.72,...,0.88,0.18,1,2048.819580,2053.041748,1811.886841,1807.664551,0.796392,0.203608,1.00
31,ARS,BOU,ARS - BOU,32,2026-04-11,3.42,1,2.21,1,36.72,...,0.88,0.18,1,2066.523193,2052.121826,1824.936646,1839.337891,0.800701,0.199299,1.00
26,ARS,WOL,WOL - ARS,27,2026-02-18,3.13,1,2.21,1,36.72,...,0.88,0.18,1,2060.096924,2053.014404,1673.604004,1681.672607,0.902457,0.097543,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
742,WOL,EVE,EVE - WOL,21,2026-01-07,1.36,14,1.02,20,19.79,...,2.00,0.40,19,1661.810425,1666.981934,1801.654663,1801.654663,0.308955,0.691045,0.97
757,WOL,BHA,BHA - WOL,36,2026-05-09,0.90,20,1.02,20,19.79,...,2.00,0.40,19,1679.114502,1679.114502,1855.705078,1861.281738,0.265702,0.734298,0.97
744,WOL,MCI,MCI - WOL,23,2026-01-24,0.93,20,1.02,20,19.79,...,2.00,0.40,19,1676.032593,1673.585815,1953.931396,1956.378174,0.168022,0.831978,0.97
758,WOL,FUL,WOL - FUL,37,2026-05-17,0.81,18,1.02,20,19.79,...,2.00,0.40,19,1673.537964,1673.537964,1801.207764,1798.762695,0.324115,0.675885,0.97


In [54]:
print(amorim_era['team'].unique())

<StringArray>
['BOU', 'LIV', 'AVL', 'NEW', 'BHA', 'FUL', 'SUN', 'WHU', 'BUR', 'TOT', 'MCI',
 'WOL', 'CHE', 'CRY', 'BRE', 'NFO', 'ARS', 'MUN', 'EVE', 'LEE']
Length: 20, dtype: str


In [55]:
selected_teams = ['ARS', 'MCI', 'AVL', 'MUN', 'LIV', 'BOU']

mun_amorim = amorim_era[amorim_era['team'].isin(selected_teams)]
mun_carrick = carrick_era[carrick_era['team'].isin(selected_teams)]


In [56]:
fig = px.line(
    mun_amorim,
    x='game_date',
    y='rolling_xg',
    text='rolling_xg',
    color='team', 
    hover_data=['rolling_xg_rank', 'game', 'xg', 'avg_xg', 'avg_xg_rank', 'avg_rolling_xg_rank']
    )

fig.update_layout(
    height=600,
    width=1200,
    title={
        "text": "Premier League Points Swing 24/25 to 25/26",
        "x": 0.5,
        "xanchor": "center",
        "yanchor": "top",
        "font": {"size": 20, "color": "#3A2A1A"},
    },
    font={
        "family": "Gill Sans, sans-serif",
        "size": 11,
        "color": "#3A2A1A",
    },
    margin={"l": 40, "r": 40, "t": 70, "b": 60},
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    legend={
        "font": {"color": "#3A2A1A"},
        "title": {"font": {"color": "#3A2A1A"}},
    },
    xaxis={
        "showgrid": False,
        "zeroline": False,
        "linecolor": "#5A4632",
        "tickcolor": "#5A4632",
        "tickfont": {"size": 13, "color": "#5A4632"},
        "title": {"font": {"color": "#3A2A1A"}, "text": "Game Date"},
    },
    yaxis={
        "showgrid": False,
        "gridcolor": "#D8C8AF",
        "zeroline": False,
        "linecolor": "#5A4632",
        "tickcolor": "#5A4632",
        "tickfont": {"size": 13, "color": "#5A4632"},
        "title": {"font": {"color": "#3A2A1A"}, "text": "Rolling xg"},
        "categoryorder": "array",
        "categoryarray": pts_swing["team"].tolist(),
    },
)
fig.show()

fig2 = px.line(
    mun_carrick,
    x='game_date',
    y='rolling_xg',
    text='rolling_xg',
    color='team',
    hover_data=['rolling_xg_rank', 'game', 'xg', 'avg_xg', 'avg_xg_rank', 'avg_rolling_xg_rank']
    )
fig2.update_layout(
    height=600,
    width=1200,
    title={
        "text": "Premier League Points Swing 24/25 to 25/26",
        "x": 0.5,
        "xanchor": "center",
        "yanchor": "top",
        "font": {"size": 20, "color": "#3A2A1A"},
    },
    font={
        "family": "Gill Sans, sans-serif",
        "size": 11,
        "color": "#3A2A1A",
    },
    margin={"l": 40, "r": 40, "t": 70, "b": 60},
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    legend={
        "font": {"color": "#3A2A1A"},
        "title": {"font": {"color": "#3A2A1A"}},
    },
    xaxis={
        "showgrid": False,
        "zeroline": False,
        "linecolor": "#5A4632",
        "tickcolor": "#5A4632",
        "tickfont": {"size": 13, "color": "#5A4632"},
        "title": {"font": {"color": "#3A2A1A"}, "text": "Game Date"},
    },
    yaxis={
        "showgrid": False,
        "gridcolor": "#D8C8AF",
        "zeroline": False,
        "linecolor": "#5A4632",
        "tickcolor": "#5A4632",
        "tickfont": {"size": 13, "color": "#5A4632"},
        "title": {"font": {"color": "#3A2A1A"}, "text": "Rolling xg"},
        "categoryorder": "array",
        "categoryarray": pts_swing["team"].tolist(),
    },
)
fig2.show()



In [57]:
team_match['rolling_xg'] = team_match.groupby('team')['xg'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True)

team_match['tot_xg'] = team_match.groupby('team')['xg'].transform(lambda x: x.sum())

team_match['avg_xg'] = team_match.groupby('team')['xg'].transform(lambda x: x.mean())

team_match['avg_rolling_xg'] = team_match.groupby('team')['rolling_xg'].transform(lambda x: x.mean())

team_match['xg_90'] = team_match.groupby('round')['tot_xg'].transform(lambda x: x / 90)

team_match['rolling_xg_rank'] = team_match.groupby('round')['rolling_xg'].rank(ascending=False)

team_match['avg_rolling_xg_rank'] = team_match.groupby('round')['avg_rolling_xg'].rank(ascending=False)

team_match['avg_xg_rank'] = team_match.groupby('round')['avg_xg'].rank(ascending=False)

#Goals Against
team_match['rolling_xga'] = team_match.groupby('team')['xga'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True)

team_match['tot_xga'] = team_match.groupby('team')['xga'].transform(lambda x: x.sum())

team_match['avg_xga'] = team_match.groupby('team')['xga'].transform(lambda x: x.mean())

team_match['xga_90'] = team_match.groupby('round')['tot_xga'].transform(lambda x: x / 90)

team_match['rolling_xga_rank'] = team_match.groupby('round')['rolling_xga'].rank(ascending=True)

team_match['avg_xga_rank'] = team_match.groupby('round')['avg_xga'].rank(ascending=True)


fig = px.line(
    team_match,
    x='game_date',
    y='rolling_xg',
    text='rolling_xg',
    color='team', 
    hover_data=['rolling_xg_rank', 'game', 'xg', 'avg_xg_rank', 'avg_rolling_xg_rank'])

fig.show()

In [64]:
fig = px.bar(
    mun_carrick,
    x='team',
    y='tilt',
    text='tilt',
    color='team', 
    )

fig.update_layout(
    height=600,
    width=1200,
    title={
        "text": "Premier League Points Swing 24/25 to 25/26",
        "x": 0.5,
        "xanchor": "center",
        "yanchor": "top",
        "font": {"size": 20, "color": "#3A2A1A"},
    },
    font={
        "family": "Gill Sans, sans-serif",
        "size": 11,
        "color": "#3A2A1A",
    },
    margin={"l": 40, "r": 40, "t": 70, "b": 60},
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    legend={
        "font": {"color": "#3A2A1A"},
        "title": {"font": {"color": "#3A2A1A"}},
    },
    xaxis={
        "showgrid": False,
        "zeroline": False,
        "linecolor": "#5A4632",
        "tickcolor": "#5A4632",
        "tickfont": {"size": 13, "color": "#5A4632"},
        "title": {"font": {"color": "#3A2A1A"}, "text": "Game Date"},
    },
    yaxis={
        "showgrid": False,
        "gridcolor": "#D8C8AF",
        "zeroline": False,
        "linecolor": "#5A4632",
        "tickcolor": "#5A4632",
        "tickfont": {"size": 13, "color": "#5A4632"},
        "title": {"font": {"color": "#3A2A1A"}, "text": "Rolling xg"},
        "categoryorder": "array",
        "categoryarray": pts_swing["team"].tolist(),
    },
)
fig.show()